In [ ]:
# NOTEBOOK 02: DATA PREPROCESSING

import os
import re

import pandas as pd
from collections import Counter

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 150)

RAW_PATH = '../data/raw'
PROCESSED_PATH = '../data/processed'
os.makedirs(PROCESSED_PATH, exist_ok=True)


In [ ]:
# ----------------------------------------------------------
# STEP 0: LOAD RAW DATA
# ----------------------------------------------------------

products = pd.read_csv(os.path.join(RAW_PATH, 'product_info.csv'))

review_files = [
    'reviews_0-250.csv',
    'reviews_250-500.csv',
    'reviews_500-750.csv',
    'reviews_750-1250.csv',
    'reviews_1250-end.csv'
]

review_dfs = []

for filename in review_files:
    review_df = pd.read_csv(
        os.path.join(RAW_PATH, filename),
        dtype={'author_id': str},
        low_memory=False
    )
    review_dfs.append(review_df)

reviews = pd.concat(review_dfs, ignore_index=True)

raw_product_rows = products.shape[0]
raw_review_rows = reviews.shape[0]

print("STEP 0: Load Raw Data")
print()
print(f"  {'Dataset':<15} {'Rows':>12} {'Columns':>10}")
print(f"  {'----------':<15} {'----------':>12} {'-------':>10}")
print(f"  {'Products':<15} {products.shape[0]:>12,} {products.shape[1]:>10}")
print(f"  {'Reviews':<15} {reviews.shape[0]:>12,} {reviews.shape[1]:>10}")


In [ ]:
# ----------------------------------------------------------
# STEP 1: SELECT RELEVANT COLUMNS
# ----------------------------------------------------------

product_cols = [
    'product_id',
    'product_name',
    'brand_name',
    'ingredients',
    'price_usd',
    'rating',
    'reviews',
    'loves_count',
    'primary_category',
    'secondary_category',
    'tertiary_category'
]

review_cols = [
    'product_id',
    'rating',
    'is_recommended',
    'review_text',
    'skin_type',
    'submission_time'
]

products_clean = products[product_cols].copy()
reviews_clean = reviews[review_cols].copy()

print("STEP 1: Select Relevant Columns")
print()
print(f"  {'Dataset':<15} {'Before':>8} {'After':>8} {'Dropped':>10}")
print(f"  {'----------':<15} {'------':>8} {'-----':>8} {'-------':>10}")
print(f"  {'Products':<15} {products.shape[1]:>8} "
      f"{products_clean.shape[1]:>8} "
      f"{products.shape[1] - products_clean.shape[1]:>10}")
print(f"  {'Reviews':<15} {reviews.shape[1]:>8} "
      f"{reviews_clean.shape[1]:>8} "
      f"{reviews.shape[1] - reviews_clean.shape[1]:>10}")


STEP 1: Select Relevant Columns

  Dataset           Before    After    Dropped
  ----------        ------    -----    -------
  Products              27       11         16
  Reviews               19        6         13


In [ ]:
# ----------------------------------------------------------
# STEP 2: HANDLE MISSING VALUES
# ----------------------------------------------------------

before_p = len(products_clean)

products_clean = products_clean.dropna(subset=['ingredients'])
dropped_p = before_p - len(products_clean)

med_rating = round(products_clean['rating'].median(), 4)
med_reviews = products_clean['reviews'].median()

products_clean['rating'] = products_clean['rating'].fillna(med_rating)
products_clean['reviews'] = products_clean['reviews'].fillna(med_reviews)

n_sec = products_clean['secondary_category'].isna().sum()
n_ter = products_clean['tertiary_category'].isna().sum()

products_clean['secondary_category'] = products_clean['secondary_category'].fillna('Unknown')
products_clean['tertiary_category'] = products_clean['tertiary_category'].fillna('Unknown')

before_r = len(reviews_clean)

reviews_clean = reviews_clean.dropna(subset=['review_text'])
dropped_r = before_r - len(reviews_clean)

mode_rec = reviews_clean['is_recommended'].mode()[0]
reviews_clean['is_recommended'] = reviews_clean['is_recommended'].fillna(mode_rec)

n_skin = reviews_clean['skin_type'].isna().sum()
reviews_clean['skin_type'] = reviews_clean['skin_type'].fillna('Unknown')

remaining_p = products_clean.isnull().sum().sum()
remaining_r = reviews_clean[['product_id', 'review_text']].isnull().sum().sum()

print("STEP 2: Handle Missing Values")
print()
print("  Products")
print()
print(f"  {'Action':<35} {'Column':<25} {'Value':>15}")
print(f"  {'----------':<35} {'------':<25} {'-----':>15}")
print(f"  {'Rows dropped (no ingredients)':<35} "
      f"{'ingredients':<25} {f'{dropped_p:,} ({dropped_p / before_p * 100:.1f}%)':>15}")
print(f"  {'Filled with median':<35} "
      f"{'rating':<25} {med_rating:>15}")
print(f"  {'Filled with median':<35} "
      f"{'reviews':<25} {med_reviews:>15,.0f}")
print(f"  {'Filled with Unknown':<35} "
      f"{'secondary_category':<25} {n_sec:>15,}")
print(f"  {'Filled with Unknown':<35} "
      f"{'tertiary_category':<25} {n_ter:>15,}")
print(f"  {'Rows remaining':<35} "
      f"{'-':<25} {len(products_clean):>15,}")

print()
print("  Reviews")
print()
print(f"  {'Action':<35} {'Column':<25} {'Value':>15}")
print(f"  {'----------':<35} {'------':<25} {'-----':>15}")
print(f"  {'Rows dropped (no review_text)':<35} "
      f"{'review_text':<25} {f'{dropped_r:,} ({dropped_r / before_r * 100:.2f}%)':>15}")
print(f"  {'Filled with mode':<35} "
      f"{'is_recommended':<25} {mode_rec:>15,.0f}")
print(f"  {'Filled with Unknown':<35} "
      f"{'skin_type':<25} {n_skin:>15,}")
print(f"  {'Rows remaining':<35} "
      f"{'-':<25} {len(reviews_clean):>15,}")

print()
print("  Verification")
print()
print(f"  {'Dataset':<25} {'Critical Missing':>18}")
print(f"  {'----------':<25} {'----------------':>18}")
print(f"  {'Products':<25} {remaining_p:>18}")
print(f"  {'Reviews':<25} {remaining_r:>18}")


STEP 2: Handle Missing Values

  Products

  Action                              Column                              Value
  ----------                          ------                              -----
  Rows dropped (no ingredients)       ingredients                   945 (11.1%)
  Filled with median                  rating                             4.2888
  Filled with median                  reviews                               144
  Filled with Unknown                 secondary_category                      4
  Filled with Unknown                 tertiary_category                     907
  Rows remaining                      -                                   7,549

  Reviews

  Action                              Column                              Value
  ----------                          ------                              -----
  Rows dropped (no review_text)       review_text                 1,444 (0.13%)
  Filled with mode                    is_recommended             

In [ ]:
# ----------------------------------------------------------
# STEP 3: REMOVE DUPLICATES
# ----------------------------------------------------------

before_p = len(products_clean)
dupes_p = products_clean.duplicated(subset=['product_id']).sum()
products_clean = products_clean.drop_duplicates(subset=['product_id'])

before_r = len(reviews_clean)
dupes_r = reviews_clean.duplicated().sum()
reviews_clean = reviews_clean.drop_duplicates()

print("STEP 3: Remove Duplicates (After Column Selection)")
print()
print(f"  {'Dataset':<15} {'Before':>10} {'Duplicates':>12} {'After':>10}")
print(f"  {'----------':<15} {'------':>10} {'----------':>12} {'-----':>10}")
print(f"  {'Products':<15} {before_p:>10,} {dupes_p:>12,} "
      f"{len(products_clean):>10,}")
print(f"  {'Reviews':<15} {before_r:>10,} {dupes_r:>12,} "
      f"{len(reviews_clean):>10,}")


STEP 3: Remove Duplicates (After Column Selection)

  Dataset             Before   Duplicates      After
  ----------          ------   ----------      -----
  Products             7,549            0      7,549
  Reviews          1,092,967          570  1,092,397


In [ ]:
# ----------------------------------------------------------
# STEP 4: CLEAN AND PARSE INGREDIENT LISTS
# ----------------------------------------------------------

def clean_ingredient_string(raw_text):
    """
    Parse raw ingredient strings into clean ingredient names.
    Handles brackets, quotes, variation headers, and parfum issues.
    """
    if pd.isna(raw_text):
        return []

    text = str(raw_text)
    text = re.sub(r"[\[\]'\"]", "", text)
    text = re.sub(r'[^,\n]+:\s*', '', text)
    text = re.sub(
        r'(parfum\s*\(fragrance\))\s+([a-zA-Z])',
        r'\1, \2',
        text,
        flags=re.IGNORECASE
    )

    parts = text.split(',')
    skip_tokens = {'nan', 'none', 'n/a', '-', '', 'and', 'or'}
    cleaned = []

    for part in parts:
        ingredient = part.strip().lower()
        ingredient = ingredient.rstrip('.;:').strip()

        if len(ingredient) < 3:
            continue
        if ingredient in skip_tokens:
            continue
        if re.match(r'^\d+\.?\d*$', ingredient):
            continue

        cleaned.append(ingredient)

    return cleaned


products_clean['ingredients_list'] = (
    products_clean['ingredients'].apply(clean_ingredient_string)
)

all_ingredients = [
    ing
    for sublist in products_clean['ingredients_list']
    for ing in sublist
]

freq = Counter(all_ingredients)
unique_ingredients = sorted(set(all_ingredients))

products_with_ing = (
    products_clean['ingredients_list'].apply(len) > 0
).sum()

avg_per_product = round(
    len(all_ingredients) / len(products_clean), 1
)

print("STEP 4: Clean and Parse Ingredient Lists")
print()
print(f"  {'Metric':<35} {'Value':>12}")
print(f"  {'----------':<35} {'-----':>12}")
print(f"  {'Products processed':<35} "
      f"{len(products_clean):>12,}")
print(f"  {'Products with ingredients':<35} "
      f"{products_with_ing:>12,}")
print(f"  {'Total ingredient mentions':<35} "
      f"{len(all_ingredients):>12,}")
print(f"  {'Unique ingredients found':<35} "
      f"{len(unique_ingredients):>12,}")
print(f"  {'Avg ingredients per product':<35} "
      f"{avg_per_product:>12}")
print()

print(f"  Top 20 Most Frequent Ingredients")
print()
print(f"  {'Rank':<6} {'Ingredient':<45} {'Count':>8}")
print(f"  {'----':<6} {'----------':<45} {'-----':>8}")

for i, (ing, count) in enumerate(freq.most_common(20), 1):
    print(f"  {i:<6} {ing:<45} {count:>8,}")

print()
print("  Sample Verification")
print()

for _, row in products_clean[
    ['product_name', 'ingredients', 'ingredients_list']
].head(3).iterrows():
    print(f"  Product : {row['product_name']}")
    print(f"  Raw     : {str(row['ingredients'])[:100]}...")
    print(f"  Cleaned : {row['ingredients_list'][:5]}")
    print()


# ----------------------------------------------------------
# STEP 4.1: REMOVE EMPTY PARSED INGREDIENT LISTS
# ----------------------------------------------------------

before_empty = len(products_clean)

products_clean = products_clean[
    products_clean['ingredients_list'].apply(len) > 0
].copy()

dropped_empty = before_empty - len(products_clean)

print("STEP 4.1: Remove Empty Parsed Ingredient Lists")
print()
print(f"  {'Rows before check':<35} {before_empty:>12,}")
print(f"  {'Rows dropped':<35} {dropped_empty:>12,}")
print(f"  {'Rows remaining':<35} {len(products_clean):>12,}")


# ----------------------------------------------------------
# STEP 4.2: INGREDIENT COUNT STATISTICS
# ----------------------------------------------------------

products_clean['ingredient_count'] = (
    products_clean['ingredients_list'].apply(len)
)

print("STEP 4.2: Ingredient Count Statistics")
print()
print(f"  {'Statistic':<25} {'Value':>15}")
print(f"  {'----------':<25} {'-----':>15}")
print(f"  {'Minimum':<25} {products_clean['ingredient_count'].min():>15}")
print(f"  {'25th percentile':<25} {products_clean['ingredient_count'].quantile(0.25):>15.0f}")
print(f"  {'Median (50th)':<25} {products_clean['ingredient_count'].median():>15.0f}")
print(f"  {'75th percentile':<25} {products_clean['ingredient_count'].quantile(0.75):>15.0f}")
print(f"  {'Maximum':<25} {products_clean['ingredient_count'].max():>15}")
print(f"  {'Mean':<25} {products_clean['ingredient_count'].mean():>15.1f}")
print(f"  {'Std Dev':<25} {products_clean['ingredient_count'].std():>15.1f}")


STEP 4: Clean and Parse Ingredient Lists

  Metric                                     Value
  ----------                                 -----
  Products processed                         7,549
  Products with ingredients                  7,544
  Total ingredient mentions                261,559
  Unique ingredients found                  13,899
  Avg ingredients per product                 34.6

  Top 20 Most Frequent Ingredients

  Rank   Ingredient                                       Count
  ----   ----------                                       -----
  1      glycerin                                         4,269
  2      phenoxyethanol                                   4,266
  3      tocopherol                                       2,979
  4      linalool                                         2,885
  5      caprylyl glycol                                  2,835
  6      limonene                                         2,813
  7      ethylhexylglycerin                         

In [ ]:
# ----------------------------------------------------------
# STEP 5: SAVE PROCESSED DATASETS
# ----------------------------------------------------------

products_clean['ingredients_parsed'] = (
    products_clean['ingredients_list']
    .apply(lambda x: ' | '.join(x))
)

products_export = products_clean.drop(columns=['ingredients_list'])

products_export.to_csv(
    os.path.join(PROCESSED_PATH, 'products_cleaned.csv'),
    index=False
)

reviews_clean.to_csv(
    os.path.join(PROCESSED_PATH, 'reviews_cleaned.csv'),
    index=False
)

print("STEP 5: Save Processed Datasets")
print()
print(f"  {'File':<30} {'Rows':>12} {'Columns':>10}")
print(f"  {'----------':<30} {'----':>12} {'-------':>10}")
print(f"  {'products_cleaned.csv':<30} "
      f"{len(products_export):>12,} "
      f"{len(products_export.columns):>10}")
print(f"  {'reviews_cleaned.csv':<30} "
      f"{len(reviews_clean):>12,} "
      f"{len(reviews_clean.columns):>10}")

print()
print("  Preprocessing Summary")
print()
print(f"  {'Metric':<35} {'Products':>12} {'Reviews':>12}")
print(f"  {'----------':<35} {'--------':>12} {'-------':>12}")
print(f"  {'Raw rows':<35} "
      f"{raw_product_rows:>12,} "
      f"{raw_review_rows:>12,}")
print(f"  {'Rows after cleaning':<35} "
      f"{len(products_export):>12,} "
      f"{len(reviews_clean):>12,}")
print(f"  {'Columns retained':<35} "
      f"{len(products_export.columns):>12} "
      f"{len(reviews_clean.columns):>12}")
print(f"  {'Unique ingredients found':<35} "
      f"{len(unique_ingredients):>12,} "
      f"{'N/A':>12}")
print()


STEP 5: Save Processed Datasets

  File                                   Rows    Columns
  ----------                             ----    -------
  products_cleaned.csv                  7,544         13
  reviews_cleaned.csv               1,092,397          6

  Preprocessing Summary

  Metric                                  Products      Reviews
  ----------                              --------      -------
  Raw rows                                   8,494    1,094,411
  Rows after cleaning                        7,544    1,092,397
  Columns retained                              13            6
  Unique ingredients found                  13,899          N/A

